In [14]:
# ============================
# 1. Importera bibliotek
# ============================

import pandas as pd
import numpy as np

# ============================
# 2. Ladda datasetet
# ============================

df = pd.read_csv("../data/telco.csv")

# Visa de första raderna
df.head()



,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [15]:
# ============================
# Rensa kolumnnamn
# Tar bort whitespace i början/slutet av kolumnnamn
# ============================

df.columns = df.columns.str.strip()

# Visa kolumnnamnen för att bekräfta
df.columns


Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='object')

In [16]:
# ============================
# Konvertera TotalCharges till numerisk
# Kolumnen innehåller tomma strängar → måste konverteras
# ============================

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Kontrollera hur många NaN som skapades
df["TotalCharges"].isnull().sum()


np.int64(11)

In [17]:
# ============================
# Hantera missing values
# Fyller NaN i TotalCharges med medianen
# ============================

df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)

# Kontrollera att inga NaN finns kvar
df.isnull().sum()

C:\Users\jimmy\AppData\Local\Temp\ipykernel_18636\4267104082.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["TotalCharges"].fillna(df["TotalCharges"].median(), inplace=True)


customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

In [18]:
# ============================
# Ta bort kolumner som inte behövs
# customerID är en identifierare → ingen prediktiv information
# ============================

df.drop("customerID", axis=1, inplace=True)

df.head()


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [19]:
# ============================
# Konvertera Churn till numerisk
# Yes → 1, No → 0
# ============================

df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

# Kontrollera resultatet
df["Churn"].value_counts()


Churn
0    5174
1    1869
Name: count, dtype: int64

In [20]:
# ============================
# Identifiera kolumntyper
# ============================

categorical_cols = df.select_dtypes(include="object").columns
numeric_cols = df.select_dtypes(include=np.number).columns

categorical_cols, numeric_cols

(Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
        'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
        'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
        'PaperlessBilling', 'PaymentMethod'],
       dtype='object'),
 Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn'], dtype='object'))

In [21]:
# ============================
# One-hot-encoding av kategoriska variabler
# drop_first=True undviker multikollinearitet
# ============================

df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

df_encoded.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,False,True,False,False,True,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,True,False,False,True,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,True,False,False,True,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,True,False,False,False,True,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,False,False,False,True,False,...,False,False,False,False,False,False,True,False,True,False


In [22]:
# ============================
# Dela upp i features (X) och target (y)
# ============================

X = df_encoded.drop("Churn", axis=1)
y = df_encoded["Churn"]

X.shape, y.shape


((7043, 30), (7043,))

In [23]:
# ============================
# Train/test-split
# stratify=y är viktigt eftersom churn är obalanserat
# ============================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

((5634, 30), (1409, 30))

In [24]:
# ============================
# Identifiera numeriska kolumner efter encoding
# (viktigt för att undvika KeyError)
# ============================

numeric_cols = X_train.select_dtypes(include=np.number).columns

numeric_cols



Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')

In [25]:
# ============================
# Skala numeriska features
# StandardScaler → bra för modeller som LR, KNN, XGBoost
# ============================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Träna scalern på träningsdata
X_train[numeric_cols] = scaler.fit_transform(X_train[numeric_cols])

# Använd samma scaler på testdata
X_test[numeric_cols] = scaler.transform(X_test[numeric_cols])

X_train.head()


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,MultipleLines_No phone service,MultipleLines_Yes,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
3738,-0.441773,0.102371,-0.521976,-0.263289,True,False,False,False,True,False,...,False,True,False,True,False,False,False,False,True,False
3151,-0.441773,-0.711743,0.337478,-0.504814,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
4860,-0.441773,-0.793155,-0.809013,-0.751213,True,True,True,False,True,False,...,False,False,False,False,False,True,False,False,False,True
3867,-0.441773,-0.263980,0.284384,-0.173699,False,True,False,True,False,False,...,False,True,False,True,False,True,True,True,False,False
3810,-0.441773,-1.281624,-0.676279,-0.990851,True,True,True,True,False,False,...,False,False,False,False,False,False,False,False,True,False


In [26]:
# ============================
# Spara preprocessade data
# ============================

import joblib

joblib.dump(X_train, "../data/X_train.pkl")
joblib.dump(X_test, "../data/X_test.pkl")
joblib.dump(y_train, "../data/y_train.pkl")
joblib.dump(y_test, "../data/y_test.pkl")


['../data/y_test.pkl']